# Text-to-SQL Agent

Ask questions about your data in plain English and let an AI agent write and run the SQL for you, all from inside your database with AIDB.

*Note: This is a simple solution on a single table. For a more complex data model, try the Semantic KB notebook.*

## What you will build

1. **A model**: a registration inside your database that points to Anthropic's Claude and uses your API key.
2. **An agent**: a named combination of instructions, a model, and a small set of tools that let it explore the database catalog and run read-only SQL.
3. **A sample table**: `public.foods`, 5,000 rows of generated food data to ask questions about.
4. **A conversation**: a first question, followed by a follow-up that only makes sense because the agent remembers the first answer.

## How it works

The agent does not guess at table or column names. When you ask a question it follows a loop:

1. Use the catalog tools to **discover** which tables and columns exist.
2. **Write** a SQL query based on what it found.
3. **Run** the query with a read-only SQL tool.
4. **Explain** the result in plain language.

Because the agent runs inside the database, every step is logged automatically. You will look at that log at the end of this notebook.

## Prerequisites

- A database with AIDB installed, reachable with the connection settings used in the *Connect to database* cell.
- An Anthropic (Claude) API key. You will be prompted for it when you run the first cell. It is held in memory only and is never written to the notebook.

## Claude API Key

Run the next cell and paste your Anthropic API key when prompted. `getpass` hides what you type so the key does not appear in the notebook output.

The key is wrapped in a small JSON document (`creds`). A later cell passes it to `aidb.create_model` as a SQL named parameter (`:creds`). The `%config SqlMagic.named_parameters = "enabled"` line is what allows `%%sql` cells to reference Python variables this way.

In [ ]:
# Provide your Claude API Key
import json
from getpass import getpass
claude_api_key = getpass("Claude API key: ")
creds = json.dumps({"api_key": claude_api_key})

%config SqlMagic.named_parameters = "enabled"

## Connect to the database

This cell creates a SQLAlchemy engine and attaches the `%sql` / `%%sql` magics to it, so every later `%%sql` cell runs against the `aidb` database.

By default it connects over the local Unix socket, which is how the demo environment is set up. If you are running against a remote database, change `PGUSER`, `PGDATABASE`, and the connection string to match your environment.

In [ ]:
from sqlalchemy import create_engine
import os, pwd

PGUSER = pwd.getpwuid(os.geteuid()).pw_name
PGPORT="5432"
PGDATABASE = os.environ.get("PGDATABASE", "aidb")

if PGUSER == "postgres":
    PGSOCKET = "/run/postgresql"
else:
    PGSOCKET = "/tmp"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}&port={PGPORT}"
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## Create the Model

A **model** in AIDB is a named, reusable connection to an LLM. Agents and SQL functions refer to it by name, so the API key and model settings live in one place.

`aidb.create_model` takes:

- `name`: the handle agents will use later (`claude_model_text_to_sql`).
- `provider`: `anthropic_messages` selects Anthropic's Messages API.
- `config`: the specific model to call. This demo uses Claude Sonnet 5. Change the model string here to try a different Claude model.
- `credentials`: the API key from the first cell. `replace_credentials => true` updates the stored key if the model already exists.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'claude_model_text_to_sql';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.models WHERE name = v_name) THEN
        PERFORM aidb.delete_model(v_name);
    END IF;

    PERFORM aidb.create_model(
        name        => v_name,
        provider    => 'anthropic_messages',
        config      => aidb.anthropic_messages_config(model => 'claude-sonnet-5'),
        credentials => CAST(:creds AS JSONB),
        replace_credentials => true
    );
END
$do$;

## Create the Agent

An **agent** bundles three things:

- **`instructions`**: the system prompt that defines its role and rules. Here the agent is told to be read-only, to discover tables before querying, and never to guess table or column names.
- **`model`**: the Claude model registered in the previous step.
- **`tools`**: the only actions the agent is allowed to take.

This agent has three tools that work together:

| Tool | What it is used for |
|---|---|
| `catalog_list_objects` | List the tables available in the database |
| `catalog_get_object_details` | Look up the columns and types of a specific table |
| `run_sql_query` | Run the SQL the agent writes and return the rows |

The agent uses the first two to find the right table and columns, then `run_sql_query` to answer your question.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'demo_agent_text_to_sql';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.agents WHERE name = v_name) THEN
        PERFORM aidb.delete_agent(v_name, force => true);
    END IF;

    PERFORM aidb.create_agent(
      name         => v_name,
      instructions => 'You are a helpful, read-only database assistant. Use catalog_list_objects and catalog_get_object_details to discover tables in the public schema, then use run_sql_query to answer. Never guess table or column names. Once you have the result, answer the question clearly and concisely.',
      model        => 'claude_model_text_to_sql',
      tools        => ARRAY['run_sql_query', 'catalog_list_objects', 'catalog_get_object_details']
    );
END
$do$;

## Create a Table

This simple table holds data about food, and it is what we will ask questions about.

- It has 5,000 rows spread evenly across eight categories: Fruit, Vegetable, Grain, Dairy, Nuts, Meat, Fish, and Snack.
- Calories and price are generated from a base value per category plus some randomness, so categories look realistic (nuts are high in calories, vegetables are low).
- `is_vegetarian` is `true` for every category except Meat and Fish.
- `ANALYZE` refreshes table statistics at the end.

Because the values are randomly generated, your numbers will differ slightly from run to run. Notice that the table is created *after* the agent. That is fine, because the agent discovers tables at question time rather than at creation time.

In [ ]:
%%sql
DROP TABLE IF EXISTS public.foods;

CREATE TABLE public.foods (
    id            SERIAL PRIMARY KEY,
    name          TEXT NOT NULL,
    category      TEXT NOT NULL,
    calories      INTEGER NOT NULL,
    price_usd     NUMERIC(6,2) NOT NULL,
    is_vegetarian BOOLEAN NOT NULL
);

INSERT INTO public.foods (name, category, calories, price_usd, is_vegetarian)
SELECT
    c.name || ' #' || g,
    c.name,
    c.base_cal + (random() * 100)::int,
    round((c.base_price + random() * 3)::numeric, 2),
    c.veg
FROM generate_series(1, 5000) AS g
CROSS JOIN LATERAL (
    SELECT * FROM (VALUES
        ('Fruit',     80, 0.50, true),
        ('Vegetable', 50, 1.00, true),
        ('Grain',    200, 0.80, true),
        ('Dairy',    300, 2.00, true),
        ('Nuts',     550, 5.00, true),
        ('Meat',     200, 3.50, false),
        ('Fish',     180, 6.00, false),
        ('Snack',    500, 2.50, true)
    ) AS t(name, base_cal, base_price, veg)
    OFFSET (g % 8) LIMIT 1
) AS c;

ANALYZE public.foods;

## Ask the Agent a Question

`aidb.agent_converse` sends a question to an agent.

In [ ]:
from sqlalchemy import text
from IPython.display import Markdown, display
import pandas as pd

def ask(agent, question, conversation_id=None):
    """Ask the agent, render the answer, return (message, conversation_id).
    Pass a conversation_id from an earlier call to ask a follow-up."""
    with conn.begin() as c:
        df = pd.read_sql(
            text("""SELECT * FROM aidb.agent_converse(
                        CAST(:agent AS text), CAST(:q AS text),
                        conversation_id => CAST(:cid AS text))"""),
            c, params={"agent": agent, "q": question, "cid": conversation_id})
    row = df.iloc[0]
    if row["error"]:
        display(Markdown(f"**Agent error:** {row['error']}"))
    else:
        display(Markdown(row["message"]))
    # On an error the function returns a NULL id, so keep the one passed in
    new_id = row["conversation_id"]
    return row["message"], (conversation_id if pd.isna(new_id) else new_id)

AGENT = "demo_agent_text_to_sql"
_, conversation_id = ask(AGENT, "What is the average calorie count of vegetarian foods by category?")

## Follow-up question using the conversation_id

Now that we have an answer, ask a follow-up. Passing the `conversation_id` tells AIDB to load the earlier messages, so the agent has the context of the previous exchange.

The follow-up, *"Which category has the least calories?"*, never says which table or which measure we mean. The agent works that out from the prior turn instead of starting from scratch. Try asking it without the `conversation_id` to see the difference.

In [ ]:
_, conversation_id = ask(AGENT, "What is the average calorie count of vegetarian foods by category?", conversation_id)

## Conversation Log

With AIDB you get more than AI in SQL: every agent interaction is recorded automatically in database tables, with no extra instrumentation.

- `aidb.conversations` holds one row per conversation.
- `aidb.conversation_log` holds the individual steps of a conversation in order, including what was asked and how the agent responded. The second query filters it to the conversation we just had.

Because the log is ordinary table data, you can audit how agents are used, debug an unexpected answer by reviewing how it was reached, or analyze usage with SQL. The *Conversations Agent* notebook builds on exactly this.

In [ ]:
%%sql
SELECT *
FROM aidb.conversations;

### Inspect the steps of one conversation

This query returns every logged step for the conversation we just ran, oldest first. Use it to see how the agent got from the question to the answer.

In [ ]:
%%sql
SELECT *
FROM aidb.conversation_log
WHERE conversation_id = :conversation_id
ORDER BY created_at;

## Recap

You created a model, an agent with three tools, and a table, and then had a multi-turn conversation with your data in plain English. Everything (the credentials, the agent definition, and the conversation history) lives in the database.

**Where to go next**

- *Semantic KB*: give the agent a searchable index of your schema so it finds the right tables in larger databases.
- *Governance*: restrict what an agent can see by tying it to a database role.
- *Conversations Agent*: analyze the conversation log you just looked at.